# A04 cifar10 — scratch

Mã huấn luyện tách từ notebook gốc, giữ tham số/thuật toán. Mặc định `RUN_TRAINING=False`: Run All chỉ kiểm imports/path và xem số liệu đã lưu; không fit/TEST. Muốn thực nghiệm mới phải tự bật cờ; mọi output legacy path được chuyển vào một thư mục `notebooks/executed/artifacts/` mới, không ghi đè kết quả gốc.


In [ ]:
# Migration r2: select this assignment's original logical paths, preserving frozen source.
from pathlib import Path as _Path
import sys as _sys
_repo = next(p for p in (_Path.cwd(), *_Path.cwd().parents) if (p / 'ASSIGN_REORGANIZATION_EXECUTION_PLAN.md').exists())
_sys.path.insert(0, str(_repo / '_shared'))
from assignment_runtime import activate as _activate
_migration = _activate('A04', 'torch')
_migration


In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    from assignment_runtime import enable_training_output_overlay
    enable_training_output_overlay()
from pathlib import Path
import json
import pandas as pd


# 1. Assignment Overview


Notebook này xây pipeline Assignment 04 cho CIFAR-10 image classification. CIFAR-10 là dataset ảnh RGB 32x32 với 10 classes, khó hơn MNIST vì object có màu, background phức tạp và variation lớn hơn.

In [ ]:
if RUN_TRAINING:
    from pathlib import Path
    import json
    import pandas as pd

    def find_project_root(start=None):
        start = Path.cwd() if start is None else Path(start)
        for path in [start.resolve(), *start.resolve().parents]:
            if (path / "datasets").exists() and (path / "pipeline04").exists():
                return path
        raise FileNotFoundError("Cannot locate project root.")

    PROJECT_ROOT = find_project_root()
    DATA_DIR = PROJECT_ROOT / "datasets" / "cifar10"
    ARTIFACT_DIR = PROJECT_ROOT / "pipeline04" / "cifar10"
    MODEL_DIR = ARTIFACT_DIR / "models"
    METRIC_DIR = ARTIFACT_DIR / "metrics"
    FIGURE_DIR = ARTIFACT_DIR / "figures"
    METADATA_DIR = ARTIFACT_DIR / "metadata"
    for directory in [DATA_DIR, MODEL_DIR, METRIC_DIR, FIGURE_DIR, METADATA_DIR]:
        directory.mkdir(parents=True, exist_ok=True)

    overview = {
        "Dataset": "CIFAR-10",
        "Source": "Canadian Institute for Advanced Research / University of Toronto archive",
        "Task": "Image classification",
        "Image representation": "32 x 32 RGB",
        "Classes": 10,
        "Notebook role": "CNN image-classification companion to MNIST",
    }
    pd.DataFrame([overview]).T.rename(columns={0: "Value"})


**Nhận xét kết quả.**

- Notebook dung CIFAR-10 tu archive chinh thuc cua University of Toronto, voi 50,000 anh train goc va 10,000 anh test.
- Representation la anh RGB kich thuoc 32 x 32 x 3, nen CNN la lua chon phu hop vi co cau truc khong gian va kenh mau that.
- Day la bai toan classification 10 lop: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck.

# 2. Environment Setup


Import thư viện, cố định seed, kiểm tra version TensorFlow/PyTorch và xác định CPU/GPU device. Output không in path cá nhân.

In [ ]:
import os
import random
import tarfile
import pickle
import time
import urllib.request
import warnings
from datetime import datetime, timezone
import matplotlib.pyplot as plt
import numpy as np
import sklearn
from IPython.display import display
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
warnings.filterwarnings('ignore', category=FutureWarning)
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
env_info = {'Python executable': Path(os.sys.executable).name, 'NumPy': np.__version__, 'Pandas': pd.__version__, 'scikit-learn': sklearn.__version__, 'Seed': SEED}
pd.DataFrame([env_info]).T.rename(columns={0: 'Value'})


**Nhận xét kết quả.**

- Device output cho biết notebook chạy trên CPU hay GPU.
- Seed được cố định để split, initialization và DataLoader reproducible hơn giữa các lần chạy.

# 3. Load Dataset


Load CIFAR-10 từ archive local trong `datasets/cifar10/`. Nếu archive chưa có, code có cơ chế tải lại từ source gốc University of Toronto.

In [ ]:
if RUN_TRAINING:
    # Explicit format adapter: the V3 archive is binary, not the historical Python pickle.
    from assignment_runtime.cifar import read_binary, CLASS_NAMES
    CIFAR10_PATH = _repo / 'tieuluan3/datasets/raw/ch3/cifar10/cifar-10-binary.tar.gz'
    x_train_full, y_train_full, x_test_raw, y_test = read_binary(CIFAR10_PATH, include_test=True)
    label_names = CLASS_NAMES
    display(pd.DataFrame({'ClassIndex': np.arange(10), 'LabelName': label_names}))
    print({'source_format': 'CIFAR10 binary; explicit adapter', 'train_shape': x_train_full.shape, 'test_shape': x_test_raw.shape})


**Nhận xét kết quả.**

- Dataset duoc doc truc tiep tu `datasets/cifar10/cifar-10-python.tar.gz`, khong extract them ban duplicate.
- Shape train/test xac nhan CIFAR-10 chuan: 50,000 train, 10,000 test, moi anh co kich thuoc 32 x 32 RGB.
- Label duoc doc tu metadata trong archive thay vi tu doan thu cong trong notebook.

# 4. Dataset Inspection


Kiểm tra class distribution, RGB channels, sample grid và một ảnh cụ thể trước preprocessing.

In [ ]:
if RUN_TRAINING:
    train_counts = np.bincount(y_train_full, minlength=10)
    test_counts = np.bincount(y_test, minlength=10)
    class_distribution = pd.DataFrame({
        "Class": np.arange(10),
        "LabelName": label_names,
        "TrainCount": train_counts,
        "TrainRate": train_counts / len(y_train_full),
        "TestCount": test_counts,
        "TestRate": test_counts / len(y_test),
    })

    sample_index = 0
    sample = x_train_full[sample_index]
    sample_info = {
        "Sample index": sample_index,
        "Shape": sample.shape,
        "Dtype": str(sample.dtype),
        "RGB channel means": np.round(sample.mean(axis=(0, 1)), 2).tolist(),
        "Min": int(sample.min()),
        "Max": int(sample.max()),
        "Label index": int(y_train_full[sample_index]),
        "Label name": label_names[int(y_train_full[sample_index])],
    }

    display(class_distribution.round(4))
    display(pd.DataFrame([sample_info]).T.rename(columns={0: "Value"}))

    fig, ax = plt.subplots(figsize=(7, 3.8))
    ax.bar(class_distribution["LabelName"], class_distribution["TrainCount"], color="#4C78A8")
    ax.set_title("CIFAR-10 Train Class Distribution")
    ax.set_xlabel("Class")
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=35)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "class_distribution.png", dpi=160)
    plt.show()

    fig, axes = plt.subplots(2, 5, figsize=(9, 4))
    for class_id, ax in enumerate(axes.ravel()):
        idx = np.where(y_train_full == class_id)[0][0]
        ax.imshow(x_train_full[idx])
        ax.set_title(label_names[class_id], fontsize=9)
        ax.axis("off")
    fig.suptitle("One CIFAR-10 Example per Class", y=1.03)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "one_example_per_class.png", dpi=160)
    plt.show()

    rng = np.random.default_rng(SEED)
    sample_ids = rng.choice(len(x_train_full), size=25, replace=False)
    fig, axes = plt.subplots(5, 5, figsize=(7, 7))
    for idx, ax in zip(sample_ids, axes.ravel()):
        ax.imshow(x_train_full[idx])
        ax.set_title(label_names[int(y_train_full[idx])], fontsize=8)
        ax.axis("off")
    fig.suptitle("Random CIFAR-10 Samples", y=1.02)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "sample_grid.png", dpi=160)
    plt.show()


**Nhận xét kết quả.**

- Class distribution của CIFAR-10 chuẩn cân bằng giữa 10 classes.
- Sample images cho thấy CIFAR-10 khó hơn MNIST: ảnh nhỏ, nhiều màu và object/background đa dạng.
- Một sample có 3 RGB channels, nên Conv2D filter học trên cả channel màu lẫn không gian ảnh.

# 5. Data Preparation


Tạo train/validation/test split, fit channel mean/std chỉ trên train, normalize dữ liệu và tạo tensor layout riêng cho Keras/PyTorch.

In [ ]:
if RUN_TRAINING:
    x_train_raw, x_valid_raw, y_train, y_valid = train_test_split(
        x_train_full,
        y_train_full,
        test_size=0.10,
        random_state=SEED,
        stratify=y_train_full,
    )

    train_mean = x_train_raw.astype("float32").mean(axis=(0, 1, 2), keepdims=True)
    train_std = x_train_raw.astype("float32").std(axis=(0, 1, 2), keepdims=True)

    def normalize_images(images):
        return ((images.astype("float32") - train_mean) / (train_std + 1e-7)).astype("float32")

    X_train_keras_full = normalize_images(x_train_raw)
    X_valid_keras_full = normalize_images(x_valid_raw)
    X_test_keras = normalize_images(x_test_raw)
    X_train_torch_full = np.transpose(X_train_keras_full, (0, 3, 1, 2))
    X_valid_torch_full = np.transpose(X_valid_keras_full, (0, 3, 1, 2))
    X_test_torch = np.transpose(X_test_keras, (0, 3, 1, 2))

    framework_train_size = 10000
    framework_valid_size = 3000
    train_idx_fw, _ = train_test_split(np.arange(len(y_train)), train_size=framework_train_size, random_state=SEED, stratify=y_train)
    valid_idx_fw, _ = train_test_split(np.arange(len(y_valid)), train_size=framework_valid_size, random_state=SEED, stratify=y_valid)

    X_train_keras = X_train_keras_full[train_idx_fw]
    y_train_fw = y_train[train_idx_fw]
    X_valid_keras = X_valid_keras_full[valid_idx_fw]
    y_valid_fw = y_valid[valid_idx_fw]
    X_train_torch = X_train_torch_full[train_idx_fw]
    X_valid_torch = X_valid_torch_full[valid_idx_fw]

    split_summary = pd.DataFrame([
        {"Split": "original train", "Images": len(y_train_full), "Shape": x_train_full.shape},
        {"Split": "train after validation split", "Images": len(y_train), "Shape": X_train_keras_full.shape},
        {"Split": "framework train budget", "Images": len(y_train_fw), "Shape": X_train_keras.shape},
        {"Split": "framework validation budget", "Images": len(y_valid_fw), "Shape": X_valid_keras.shape},
        {"Split": "test", "Images": len(y_test), "Shape": X_test_keras.shape},
    ])
    normalization_summary = pd.DataFrame({
        "Channel": ["R", "G", "B"],
        "TrainMean": train_mean.reshape(-1),
        "TrainStd": train_std.reshape(-1),
    })
    layout_summary = pd.DataFrame([
        {"Framework": "Keras/TensorFlow", "Layout": "B x H x W x C", "TrainShape": X_train_keras.shape},
        {"Framework": "PyTorch", "Layout": "B x C x H x W", "TrainShape": X_train_torch.shape},
    ])

    display(split_summary)
    display(normalization_summary.round(4))
    display(layout_summary)


**Nhận xét kết quả.**

- Original train split duoc tach thanh 45,000 train va 5,000 validation truoc khi lay subset runtime.
- Framework training dung 10,000 train va 3,000 validation de notebook chay duoc tren CPU/laptop.
- Channel mean/std duoc fit tren train split, sau do ap dung cho validation/test; test set van giu doc lap.

# 6. Discover CNN Operations


Minh họa Conv2D, ReLU, MaxPool, Flatten, Dense logits và softmax bằng tensor RGB nhỏ.

**Nhận xét kết quả.**

- Input có 3 channels RGB; mỗi Conv2D filter học weights trên cả 3 channels và tạo một feature map.
- Pooling giảm spatial size, Flatten chuyển feature maps sang vector, Dense tạo 10 logits.

# 7. Inspect CNN Building Blocks


Kiểm tra Basic CNN block, residual block và SE/channel-attention block bằng dummy tensor trước khi train model thật.

**Nhận xét kết quả.**

- Residual block đổi channels 3→16 và giảm spatial size bằng stride=2, nên shortcut dùng projection để shape khớp.
- SE block giữ nguyên tensor shape nhưng tạo channel weights để scale từng feature channel.

# 8. CNN From Scratch with NumPy


Định nghĩa NumPy CNN layers có forward/backward thật cho RGB CIFAR-10: Conv2D, ReLU, MaxPool, Flatten, Dense, SoftmaxCrossEntropy và SGD update.

In [ ]:
if RUN_TRAINING:
    def im2col_nhwc(x, kh, kw, pad=0, stride=1):
        x_pad = np.pad(x, ((0, 0), (pad, pad), (pad, pad), (0, 0)), mode="constant")
        b, h, w, c = x_pad.shape
        out_h = (h - kh) // stride + 1
        out_w = (w - kw) // stride + 1
        cols = np.empty((x.shape[0], out_h, out_w, kh, kw, c), dtype=x.dtype)
        for i in range(kh):
            for j in range(kw):
                cols[:, :, :, i, j, :] = x_pad[:, i:i + out_h * stride:stride, j:j + out_w * stride:stride, :]
        return cols.reshape(x.shape[0] * out_h * out_w, kh * kw * c), x_pad.shape, out_h, out_w

    def col2im_nhwc(cols, x_shape, x_pad_shape, kh, kw, pad=0, stride=1):
        b, h, w, c = x_shape
        _, hp, wp, _ = x_pad_shape
        out_h = (hp - kh) // stride + 1
        out_w = (wp - kw) // stride + 1
        cols = cols.reshape(b, out_h, out_w, kh, kw, c)
        dx_pad = np.zeros(x_pad_shape, dtype=cols.dtype)
        for i in range(kh):
            for j in range(kw):
                dx_pad[:, i:i + out_h * stride:stride, j:j + out_w * stride:stride, :] += cols[:, :, :, i, j, :]
        return dx_pad[:, pad:pad + h, pad:pad + w, :] if pad else dx_pad

    class Conv2DNumPy:
        def __init__(self, in_channels, out_channels, kernel_size=3, padding=1, seed=0):
            rng = np.random.default_rng(seed)
            scale = np.sqrt(2.0 / (kernel_size * kernel_size * in_channels))
            self.W = rng.normal(0, scale, (kernel_size, kernel_size, in_channels, out_channels)).astype("float32")
            self.b = np.zeros(out_channels, dtype="float32")
            self.padding = padding
            self.kernel_size = kernel_size
        def forward(self, x):
            self.x = x
            self.cols, self.x_pad_shape, out_h, out_w = im2col_nhwc(x, self.kernel_size, self.kernel_size, self.padding)
            out = self.cols @ self.W.reshape(-1, self.W.shape[-1]) + self.b
            return out.reshape(x.shape[0], out_h, out_w, self.W.shape[-1])
        def backward(self, dout):
            dout2 = dout.reshape(-1, dout.shape[-1])
            self.dW = (self.cols.T @ dout2).reshape(self.W.shape)
            self.db = dout2.sum(axis=0)
            dcols = dout2 @ self.W.reshape(-1, self.W.shape[-1]).T
            return col2im_nhwc(dcols, self.x.shape, self.x_pad_shape, self.kernel_size, self.kernel_size, self.padding)
        def update(self, lr):
            self.W -= lr * self.dW
            self.b -= lr * self.db

    class ReLUNumPy:
        def forward(self, x):
            self.mask = x > 0
            return x * self.mask
        def backward(self, dout):
            return dout * self.mask
        def update(self, lr):
            pass

    class MaxPool2DNumPy:
        def __init__(self, pool=2, stride=2):
            self.pool = pool
            self.stride = stride
        def forward(self, x):
            self.x = x
            b, h, w, c = x.shape
            out_h, out_w = h // self.stride, w // self.stride
            out = np.empty((b, out_h, out_w, c), dtype=x.dtype)
            self.argmax = np.empty((b, out_h, out_w, c), dtype=np.int32)
            for i in range(out_h):
                for j in range(out_w):
                    window = x[:, i*self.stride:i*self.stride+self.pool, j*self.stride:j*self.stride+self.pool, :]
                    flat = window.reshape(b, self.pool * self.pool, c)
                    self.argmax[:, i, j, :] = flat.argmax(axis=1)
                    out[:, i, j, :] = flat.max(axis=1)
            return out
        def backward(self, dout):
            dx = np.zeros_like(self.x)
            b, out_h, out_w, c = dout.shape
            for i in range(out_h):
                for j in range(out_w):
                    for n in range(b):
                        for ch in range(c):
                            pos = self.argmax[n, i, j, ch]
                            dx[n, i*self.stride + pos // self.pool, j*self.stride + pos % self.pool, ch] += dout[n, i, j, ch]
            return dx
        def update(self, lr):
            pass

    class FlattenNumPy:
        def forward(self, x):
            self.shape = x.shape
            return x.reshape(x.shape[0], -1)
        def backward(self, dout):
            return dout.reshape(self.shape)
        def update(self, lr):
            pass

    class DenseNumPy:
        def __init__(self, in_features, out_features, seed=0):
            rng = np.random.default_rng(seed)
            self.W = rng.normal(0, np.sqrt(2.0 / in_features), (in_features, out_features)).astype("float32")
            self.b = np.zeros(out_features, dtype="float32")
        def forward(self, x):
            self.x = x
            return x @ self.W + self.b
        def backward(self, dout):
            self.dW = self.x.T @ dout
            self.db = dout.sum(axis=0)
            return dout @ self.W.T
        def update(self, lr):
            self.W -= lr * self.dW
            self.b -= lr * self.db

    class SoftmaxCrossEntropyNumPy:
        def forward(self, logits, labels):
            shifted = logits - logits.max(axis=1, keepdims=True)
            exp = np.exp(shifted)
            self.probs = exp / exp.sum(axis=1, keepdims=True)
            self.labels = labels
            return -np.log(self.probs[np.arange(len(labels)), labels] + 1e-9).mean()
        def backward(self):
            grad = self.probs.copy()
            grad[np.arange(len(self.labels)), self.labels] -= 1
            return grad / len(self.labels)

    class SGDNumPy:
        def __init__(self, lr):
            self.lr = lr
        def step(self, layers):
            for layer in layers:
                layer.update(self.lr)

    print("NumPy CIFAR-10 CNN components defined:", 8)


**Nhận xét kết quả.**

- NumPy components có forward/backward thật cho Conv2D, ReLU, MaxPool2D, Flatten, Dense và SoftmaxCrossEntropy.
- `SGDNumPy` là optimizer update thủ công; phần này không dùng keras layers, torch layers hay autograd.
- Conv2D nhận RGB input nên weight shape bao gồm cả 3 input channels.

# 9. NumPy CNN Training


Test từng layer NumPy, train một CNN nhỏ trên stratified CIFAR-10 subset và lưu best state.

In [ ]:
if RUN_TRAINING:
    def macro_metrics(y_true, y_pred):
        return {
            "Accuracy": accuracy_score(y_true, y_pred),
            "Macro-P": precision_score(y_true, y_pred, average="macro", zero_division=0),
            "Macro-R": recall_score(y_true, y_pred, average="macro", zero_division=0),
            "Macro-F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        }

    class SimpleNumPyCIFARCNN:
        def __init__(self, seed=SEED):
            self.layers = [
                Conv2DNumPy(3, 6, 3, 1, seed + 1), ReLUNumPy(), MaxPool2DNumPy(),
                Conv2DNumPy(6, 12, 3, 1, seed + 2), ReLUNumPy(), MaxPool2DNumPy(),
                FlattenNumPy(), DenseNumPy(8 * 8 * 12, 64, seed + 3), ReLUNumPy(), DenseNumPy(64, 10, seed + 4),
            ]
            self.loss_fn = SoftmaxCrossEntropyNumPy()
        def forward(self, x, collect_shapes=False):
            shapes = [("Input", x.shape)]
            out = x
            for layer in self.layers:
                out = layer.forward(out)
                if collect_shapes:
                    shapes.append((layer.__class__.__name__, out.shape))
            return (out, shapes) if collect_shapes else out
        def backward(self, grad):
            for layer in reversed(self.layers):
                grad = layer.backward(grad)
        def update(self, lr):
            SGDNumPy(lr).step(self.layers)
        def train_batch(self, x, y, lr):
            logits = self.forward(x)
            loss = self.loss_fn.forward(logits, y)
            self.backward(self.loss_fn.backward())
            self.update(lr)
            return float(loss)
        def predict_proba(self, x, batch_size=256):
            probs = []
            for start in range(0, len(x), batch_size):
                logits = self.forward(x[start:start + batch_size])
                shifted = logits - logits.max(axis=1, keepdims=True)
                exp = np.exp(shifted)
                probs.append(exp / exp.sum(axis=1, keepdims=True))
            return np.vstack(probs)
        def state_dict(self):
            return {f"{i}_{name}": getattr(layer, name).copy() for i, layer in enumerate(self.layers) if hasattr(layer, "W") for name in ["W", "b"]}
        def load_state_dict(self, state):
            for i, layer in enumerate(self.layers):
                if hasattr(layer, "W"):
                    layer.W[...] = state[f"{i}_W"]
                    layer.b[...] = state[f"{i}_b"]
        def parameter_count(self):
            return int(sum(layer.W.size + layer.b.size for layer in self.layers if hasattr(layer, "W")))

    component_tests = []
    rng_test = np.random.default_rng(SEED + 100)
    unit_x = rng_test.normal(size=(2, 5, 5, 3)).astype("float32")
    unit_grad = rng_test.normal(size=(2, 5, 5, 2)).astype("float32")

    conv_unit = Conv2DNumPy(3, 2, 3, 1, seed=SEED + 101)
    conv_out = conv_unit.forward(unit_x)
    conv_dx = conv_unit.backward(unit_grad)
    component_tests.extend([
        {"Check": "Conv2D output shape", "Value": conv_out.shape},
        {"Check": "Conv2D input grad shape", "Value": conv_dx.shape},
        {"Check": "Conv2D parameter shapes", "Value": (conv_unit.W.shape, conv_unit.b.shape)},
        {"Check": "Conv2D finite output/grad", "Value": bool(np.isfinite(conv_out).all() and np.isfinite(conv_dx).all())},
    ])

    old_weight = float(conv_unit.W[0, 0, 0, 0])
    eps = 1e-3
    conv_unit.W[0, 0, 0, 0] = old_weight + eps
    plus = float((conv_unit.forward(unit_x) * unit_grad).sum())
    conv_unit.W[0, 0, 0, 0] = old_weight - eps
    minus = float((conv_unit.forward(unit_x) * unit_grad).sum())
    conv_unit.W[0, 0, 0, 0] = old_weight
    conv_unit.forward(unit_x)
    conv_unit.backward(unit_grad)
    conv_numeric_grad = (plus - minus) / (2 * eps)
    component_tests.append({"Check": "Conv2D numerical grad close", "Value": bool(np.isclose(conv_numeric_grad, conv_unit.dW[0, 0, 0, 0], atol=2e-2))})

    relu_unit = ReLUNumPy()
    relu_out = relu_unit.forward(conv_out)
    relu_dx = relu_unit.backward(np.ones_like(relu_out))
    component_tests.extend([
        {"Check": "ReLU output shape", "Value": relu_out.shape},
        {"Check": "ReLU grad shape", "Value": relu_dx.shape},
    ])

    pool_unit = MaxPool2DNumPy()
    pool_out = pool_unit.forward(relu_out)
    pool_dx = pool_unit.backward(np.ones_like(pool_out))
    component_tests.extend([
        {"Check": "MaxPool output shape", "Value": pool_out.shape},
        {"Check": "MaxPool grad shape", "Value": pool_dx.shape},
    ])

    flat_unit = FlattenNumPy()
    flat_out = flat_unit.forward(pool_out)
    flat_dx = flat_unit.backward(np.ones_like(flat_out))
    component_tests.extend([
        {"Check": "Flatten output shape", "Value": flat_out.shape},
        {"Check": "Flatten grad shape", "Value": flat_dx.shape},
    ])

    dense_unit = DenseNumPy(flat_out.shape[1], 10, seed=SEED + 102)
    dense_out = dense_unit.forward(flat_out)
    dense_dx = dense_unit.backward(np.ones_like(dense_out) / dense_out.size)
    component_tests.extend([
        {"Check": "Dense output shape", "Value": dense_out.shape},
        {"Check": "Dense grad/input shape", "Value": dense_dx.shape},
        {"Check": "Dense parameter shapes", "Value": (dense_unit.W.shape, dense_unit.b.shape)},
    ])

    loss_unit = SoftmaxCrossEntropyNumPy()
    loss_value = loss_unit.forward(dense_out, np.array([0, 1], dtype="int64"))
    loss_grad = loss_unit.backward()
    component_tests.extend([
        {"Check": "SoftmaxCE loss finite", "Value": bool(np.isfinite(loss_value))},
        {"Check": "Softmax sums approx 1", "Value": bool(np.allclose(loss_unit.probs.sum(axis=1), 1.0, atol=1e-6))},
        {"Check": "Loss grad logits shape", "Value": loss_grad.shape},
    ])

    model_np = SimpleNumPyCIFARCNN()
    test_x = X_train_keras[:4].astype("float32")
    test_y = y_train_fw[:4].astype("int64")
    logits, shape_trace = model_np.forward(test_x, collect_shapes=True)
    loss = model_np.loss_fn.forward(logits, test_y)
    model_np.backward(model_np.loss_fn.backward())
    layer_tests = component_tests.copy()
    layer_tests.extend([
        {"Check": "Logits shape", "Value": logits.shape},
        {"Check": "Loss finite", "Value": bool(np.isfinite(loss))},
        {"Check": "Softmax row sum", "Value": float(model_np.loss_fn.probs[0].sum())},
    ])
    for idx, layer in enumerate(model_np.layers):
        if hasattr(layer, "dW"):
            layer_tests.append({"Check": f"Layer {idx} grad W shape", "Value": layer.dW.shape})

    for name, value in model_np.state_dict().items():
        layer_tests.append({"Check": f"Parameter {name} shape", "Value": value.shape})

    tiny_model = SimpleNumPyCIFARCNN(seed=SEED + 200)
    tiny_x = X_train_keras[:32].astype("float32")
    tiny_y = y_train_fw[:32].astype("int64")
    initial_tiny_loss = tiny_model.loss_fn.forward(tiny_model.forward(tiny_x), tiny_y)
    for _ in range(10):
        tiny_model.train_batch(tiny_x, tiny_y, 0.005)
    final_tiny_loss = tiny_model.loss_fn.forward(tiny_model.forward(tiny_x), tiny_y)
    layer_tests.append({"Check": "Tiny batch initial loss", "Value": float(initial_tiny_loss)})
    layer_tests.append({"Check": "Tiny batch final loss", "Value": float(final_tiny_loss)})
    layer_tests.append({"Check": "Tiny batch loss decreases", "Value": bool(final_tiny_loss <= initial_tiny_loss)})

    display(pd.DataFrame(layer_tests))
    display(pd.DataFrame(shape_trace, columns=["Stage", "Shape"]))

    numpy_train_size = 4000
    numpy_valid_size = 1000
    idx_train_np, _ = train_test_split(np.arange(len(y_train)), train_size=numpy_train_size, random_state=SEED, stratify=y_train)
    idx_valid_np, _ = train_test_split(np.arange(len(y_valid)), train_size=numpy_valid_size, random_state=SEED, stratify=y_valid)
    X_np_train = X_train_keras_full[idx_train_np].astype("float32")
    y_np_train = y_train[idx_train_np].astype("int64")
    X_np_valid = X_valid_keras_full[idx_valid_np].astype("float32")
    y_np_valid = y_valid[idx_valid_np].astype("int64")

    batch_size_np = 128
    epochs_np = 2
    lr_np = 0.01
    history_np = []
    best_np = {"Macro-F1": -1, "state": None, "epoch": 0}
    rng = np.random.default_rng(SEED)
    start_time = time.perf_counter()
    for epoch in range(1, epochs_np + 1):
        order = rng.permutation(len(X_np_train))
        losses = []
        for start in range(0, len(order), batch_size_np):
            ids = order[start:start + batch_size_np]
            losses.append(model_np.train_batch(X_np_train[ids], y_np_train[ids], lr_np))
        val_prob = model_np.predict_proba(X_np_valid)
        val_pred = val_prob.argmax(axis=1)
        metrics = macro_metrics(y_np_valid, val_pred)
        row = {"Epoch": epoch, "TrainLoss": float(np.mean(losses)), **metrics}
        history_np.append(row)
        if metrics["Macro-F1"] > best_np["Macro-F1"]:
            best_np = {"Macro-F1": metrics["Macro-F1"], "state": model_np.state_dict(), "epoch": epoch}

    numpy_train_time = time.perf_counter() - start_time
    model_np.load_state_dict(best_np["state"])
    np.savez(MODEL_DIR / "numpy_cnn_best_state.npz", **best_np["state"])
    numpy_history = pd.DataFrame(history_np)
    numpy_history.to_csv(METRIC_DIR / "numpy_cnn_history.csv", index=False)

    numpy_training_summary = {
        "Train samples": numpy_train_size,
        "Validation samples": numpy_valid_size,
        "Epochs": epochs_np,
        "Batch size": batch_size_np,
        "Optimizer": "SGD",
        "Learning rate": lr_np,
        "Best epoch": best_np["epoch"],
        "Train time": numpy_train_time,
        "Parameters": model_np.parameter_count(),
    }
    display(pd.DataFrame([numpy_training_summary]).T.rename(columns={0: "Value"}))
    display(numpy_history.round(4))


**Nhận xét kết quả.**

- Layer tests kiểm tra output shape, gradient shape, finite values, parameter shape và softmax row sums trước khi train.
- Numerical gradient check trên Conv2D xác nhận backward hợp lý trong một case nhỏ.
- NumPy CNN train thật trên subset 4,000 ảnh; best state được chọn theo validation Macro-F1 và lưu vào artifact.

# 10. Tensor Shape and Parameter Inspection


Tính parameter từng layer của NumPy CNN và kiểm tra manual calculation cho Conv layer đầu.

In [ ]:
if RUN_TRAINING:
    param_rows = []
    for idx, layer in enumerate(model_np.layers):
        if hasattr(layer, "W"):
            param_rows.append({
                "LayerIndex": idx,
                "Layer": layer.__class__.__name__,
                "WeightShape": layer.W.shape,
                "BiasShape": layer.b.shape,
                "Parameters": int(layer.W.size + layer.b.size),
            })
    param_table = pd.DataFrame(param_rows)
    display(param_table)
    print("Total NumPy CNN parameters:", int(param_table["Parameters"].sum()))
    print("Manual Conv1 parameters:", (3 * 3 * 3 * 6) + 6)


**Nhận xét kết quả.**

- Conv1 dùng kernel 3x3, 3 RGB input channels và 6 filters nên có `(3x3x3x6)+6` parameters.
- Parameter count nhỏ giúp NumPy CNN chạy được trên CPU nhưng vẫn thể hiện đúng CNN building blocks.

# 11. CNN with Keras/TensorFlow


Xây Keras Basic CNN cho CIFAR-10 với augmentation chỉ áp dụng trong training.

In [ ]:
if RUN_TRAINING:
    def save_json(path, payload):
        with Path(path).open('w', encoding='utf-8') as f:
            json.dump(payload, f, ensure_ascii=False, indent=2)

    def evaluate_predictions(y_true, y_pred):
        return {'Accuracy': accuracy_score(y_true, y_pred), 'Macro-P': precision_score(y_true, y_pred, average='macro', zero_division=0), 'Macro-R': recall_score(y_true, y_pred, average='macro', zero_division=0), 'Macro-F1': f1_score(y_true, y_pred, average='macro', zero_division=0)}


**Nhận xét kết quả.**

- Keras Basic CNN nhận `(B, 32, 32, 3)` và trả `(B, 10)` probabilities.
- Augmentation sẽ được đặt trong `tf.data` training pipeline ở cell sau để validation/test không đổi shape và không bị augment.

# 12. Keras Training


Train Keras Basic CNN với checkpoint, EarlyStopping và lưu training curves.

In [ ]:
if RUN_TRAINING:
    BATCH_SIZE = 256
    MAX_EPOCHS = 3


**Nhận xét kết quả.**

- Training dùng augmentation trên train only và validation data không augment.
- Curves giúp xem CIFAR-10 có underfit/overfit trong budget epoch ngắn hay không.

# 13. Keras Evaluation


Evaluate Keras Basic CNN trên full CIFAR-10 test set và lưu confusion matrix.

**Nhận xét kết quả.**

- Keras Basic CNN đạt Accuracy 50.09% và Macro-F1 49.84% trên full CIFAR-10 test set.
- Confusion matrix cho thấy model đã học được tín hiệu ảnh RGB, nhưng CIFAR-10 vẫn khó hơn MNIST nên lỗi giữa các lớp gần nhau còn rõ.
- Metrics được báo cáo theo macro average để mỗi class có trọng số ngang nhau.

# 14. CNN with PyTorch


Xây PyTorch Basic CNN tương đương ý tưởng với Keras và kiểm tra batch/logits shape trước training.

**Nhận xét kết quả.**

- PyTorch nhận CIFAR-10 theo layout `(B, 3, 32, 32)`.
- Logits shape `(B, 10)` cho thấy mỗi ảnh có 10 class scores trước softmax.

# 15. PyTorch Training


Train PyTorch Basic CNN bằng training loop rõ ràng và dùng augmentation chỉ trong train loop.

**Nhận xét kết quả.**

- Training loop có đủ `zero_grad`, forward, loss, backward và optimizer step.
- Augmentation chỉ nằm trong train loop; validation/test đi qua ảnh normalized không augment.

# 16. PyTorch Evaluation


Load PyTorch Basic CNN checkpoint tốt nhất và evaluate trên full test set.

**Nhận xét kết quả.**

- PyTorch Basic CNN dat Accuracy 49.62% va Macro-F1 49.55% voi architecture tuong duong y tuong voi Keras baseline.
- Loop train hien ro cac buoc `zero_grad`, forward logits, loss, backward va optimizer step.
- Khac biet so voi Keras trong mot lan chay khong du de ket luan framework nao tot hon; backend, augmentation va CPU runtime deu anh huong.

# 17. Framework Comparison


Evaluate NumPy CNN trên full test set và so sánh NumPy, Keras Basic CNN, PyTorch Basic CNN.

In [ ]:
if RUN_TRAINING:
    start = time.perf_counter()
    numpy_test_prob = model_np.predict_proba(X_test_keras.astype("float32"), batch_size=256)
    numpy_pred = numpy_test_prob.argmax(axis=1)
    numpy_prediction_time = time.perf_counter() - start
    numpy_metrics = macro_metrics(y_test, numpy_pred)
    numpy_cm = confusion_matrix(y_test, numpy_pred)
    numpy_result = {
        "Model": "NumPy CNN",
        **numpy_metrics,
        "Params": int(model_np.parameter_count()),
        "Train Samples": int(numpy_train_size),
        "Best Epoch": int(best_np["epoch"]),
        "Train Time": float(numpy_train_time),
        "Pred Time": float(numpy_prediction_time),
    }
    save_json(METRIC_DIR / "numpy_cnn_test_metrics.json", numpy_result)

    fig, ax = plt.subplots(figsize=(6, 5.5))
    im = ax.imshow(numpy_cm, cmap="Purples")
    ax.set_title("NumPy CNN Confusion Matrix")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_xticks(range(10), label_names, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(10), label_names, fontsize=8)
    fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "numpy_confusion_matrix.png", dpi=160)
    plt.show()

    framework_comparison = pd.DataFrame([numpy_result, keras_result, torch_result])
    framework_comparison.to_csv(METRIC_DIR / "framework_comparison.csv", index=False)
    display(framework_comparison.round(4))


**Nhận xét kết quả.**

- Framework comparison tách riêng ba cách cài đặt cùng ý tưởng CNN cơ bản: NumPy, Keras và PyTorch.
- Keras Basic CNN có Macro-F1 49.84%, PyTorch Basic CNN có Macro-F1 49.55%, NumPy CNN có Macro-F1 25.65%.
- NumPy dùng 4,000 mẫu train, trong khi Keras/PyTorch dùng 10,000 mẫu; runtime phụ thuộc backend/hardware.

# 18. CNN Architecture Improvements


Dùng PyTorch làm framework chính để so M1 Basic CNN, M2 BatchNorm CNN, M3 Residual CNN và M4 Residual CNN + SE attention.

In [ ]:
if RUN_TRAINING:
    architecture_results = []
    trained_arch_models = {}


**Nhận xét kết quả.**

- M1-M4 dùng cùng split, normalization, augmentation, batch size, optimizer family và test set để báo cáo metrics.
- Bảng `selection_results` dùng validation Macro-F1 để chọn model, tách khỏi bảng test metrics.
- M3/M4 có residual connections; M4 thêm SE channel attention nhưng không được mặc định là tốt nhất trước khi xem output.

# 19. Improved Model Comparison


Kết hợp framework comparison và architecture evolution để chọn final candidate.

**Nhận xét kết quả.**

- Trong nhóm M1-M4, model tốt nhất là `M4 Residual CNN + SE Attention` với Macro-F1 47.39%.
- Bảng all-model cho thấy model đứng đầu toàn bộ run hiện tại là `Keras Basic CNN` với Macro-F1 49.84%.
- Kết quả không mặc định attention luôn tốt hơn; với budget ngắn, M4 tốt nhất trong nhóm architecture nhưng chưa vượt Keras Basic CNN.

# 20. Feature Map Inspection


Hiển thị original CIFAR images và feature maps từ early/deeper block của final PyTorch architecture candidate.

**Nhận xét kết quả.**

- Feature maps duoc lay tu M4 Residual CNN + SE Attention va luu thanh hinh `feature_maps.png`.
- Cac block som giu nhieu chi tiet cuc bo hon, trong khi block sau co spatial size nho hon va channel representation giau hon.
- Notebook khong gan nhan tuyet doi tung channel la mot detector cu the, vi feature map can duoc doc nhu tin hieu trung gian cua model.

# 21. Error Analysis


Tạo normalized confusion matrix, per-class accuracy và high-confidence mistakes của final candidate.

In [ ]:
if RUN_TRAINING:
    confusion_pairs = []


**Nhận xét kết quả.**

- Error analysis dùng final model đã được chọn bằng validation Macro-F1, rồi phân tích lỗi trên test set.
- Confusion matrix chuẩn hóa và per-class accuracy cho biết class nào yếu trong run hiện tại.
- High-confidence mistakes giúp xem những trường hợp model sai nhưng vẫn tự tin.

# 22. Final Model


Chọn final model theo Macro-F1, lưu model cuối và final metrics.

**Nhận xét kết quả.**

- Final model được chọn bằng validation Macro-F1, sau đó mới đọc test metrics tương ứng để báo cáo.
- Test set không được dùng để tune/chọn model; nó chỉ đóng vai trò đánh giá cuối.
- Model final đã được lưu trong `pipeline04/cifar10/models/` để notebook/report sau có thể tải lại.

# 23. Final Results


Lưu benchmark, metadata và liệt kê artifacts CIFAR-10 CNN.

In [ ]:
if RUN_TRAINING:
    artifact_listing = []


**Nhận xét kết quả.**

- Bảng tổng hợp xếp 7 model theo Macro-F1; model đứng đầu là `Keras Basic CNN` với Macro-F1 49.84%.
- Tổng train time được ghi nhận trong artifacts là 256.3s; clean run-all CIFAR-10 cuối cùng mất 330.80s trong môi trường hiện tại.
- Artifacts đã gồm 8 model files, 26 metrics files và 13 figures.

In [ ]:
from IPython.display import display
import pandas as pd
_saved = _repo / 'A04/artifacts/pipeline04/cifar10/metrics'
for _file in sorted(_saved.glob('*comparison*.csv')):
    print(_file.name)
    display(pd.read_csv(_file))
